# 00 Master End-to-End Pipeline

Single notebook to run full Rail Video Ops pipeline.

This notebook does:
- config load
- GPU check
- single video run
- batch run (raw folder)
- inline demo video preview (lanes, marker lines, boxes, IDs)
- Google Sheets sync status
- optional delete-on-success behavior

All heavy logic still comes from shared Python modules.

In [ ]:
from pathlib import Path
import sys
import json
from datetime import datetime

sys.path.append(str(Path.cwd() / 'src'))

import pandas as pd
from IPython.display import Video, display

from rail_video_intelligence.pipeline import (
    RailVideoPipeline,
    SheetsConfig,
    load_camera_profile,
    load_pipeline_settings,
    parse_pipeline_settings,
    summarize_results,
)


In [ ]:
# ===== USER CONTROLS =====
PIPELINE_CONFIG_PATH = Path('configs/pipeline.yaml')
CAMERA_PROFILE_PATH = Path('configs/camera_profiles/camera_01.yaml')
RAW_DIR = Path('data/raw')
OUTPUT_ROOT_OVERRIDE = Path('outputs/v1')

# one-video target
ONE_VIDEO = RAW_DIR / 'camera_01_2026_04_09_233235.MOV'

# behavior toggles
RUN_SINGLE = True
RUN_BATCH = False
SHOW_INLINE_DEMO_VIDEO = True
DELETE_ON_SUCCESS = False
EXPORT_EXCEL = True
FORCE_GPU = True

# live popup window (only works in local desktop environment, usually not in headless Colab runtime)
SHOW_LIVE_POPUP = False

In [ ]:
# GPU availability check
try:
    import torch
    gpu_available = torch.cuda.is_available()
    gpu_name = torch.cuda.get_device_name(0) if gpu_available else None
except Exception:
    gpu_available = False
    gpu_name = None

print('GPU available:', gpu_available)
print('GPU name:', gpu_name)


In [ ]:
# Load configs and build pipeline object
raw_cfg = load_pipeline_settings(PIPELINE_CONFIG_PATH)
pipeline_cfg = raw_cfg.get('pipeline', raw_cfg)

if FORCE_GPU and gpu_available:
    pipeline_cfg['device'] = 'cuda:0'
elif FORCE_GPU and not gpu_available:
    print('FORCE_GPU=True but no GPU found. Falling back to config/default device.')

pipeline_cfg['show_live'] = SHOW_LIVE_POPUP
pipeline_cfg['save_demo_overlay'] = True

settings = parse_pipeline_settings(pipeline_cfg)
profile = load_camera_profile(CAMERA_PROFILE_PATH)

sheets_raw = raw_cfg.get('sheets', {})
sheets_cfg = SheetsConfig(
    enabled=bool(sheets_raw.get('enabled', False)),
    spreadsheet_id=sheets_raw.get('spreadsheet_id'),
    worksheet_name=sheets_raw.get('worksheet_name', 'train_events'),
    credentials_path=Path(sheets_raw['credentials_path']) if sheets_raw.get('credentials_path') else None,
    retries=int(sheets_raw.get('retries', 4)),
    backoff_seconds=float(sheets_raw.get('backoff_seconds', 1.5)),
)

pipeline = RailVideoPipeline(
    settings=settings,
    camera_profile=profile,
    output_root=OUTPUT_ROOT_OVERRIDE,
    sheets_config=sheets_cfg,
)

print('Pipeline ready')
print('Device:', settings.device or 'auto')
print('Mode:', settings.mode)
print('Sheets enabled:', sheets_cfg.enabled)
print('Sheet ID:', sheets_cfg.spreadsheet_id)
print('Sheet tab:', sheets_cfg.worksheet_name)


In [ ]:
# Run ONE video
single_result = None
if RUN_SINGLE:
    assert ONE_VIDEO.exists(), f'Missing video: {ONE_VIDEO}'
    run_name = f'master_single_{ONE_VIDEO.stem}_{datetime.utcnow().strftime("%Y%m%d_%H%M%S")}'
    single_result = pipeline.process_video(
        video_path=ONE_VIDEO,
        run_name=run_name,
        delete_on_success=DELETE_ON_SUCCESS,
        export_excel=EXPORT_EXCEL,
        show_live=SHOW_LIVE_POPUP,
    )
    print('Single run complete:', run_name)
    print('Summary JSON:', single_result.output_json_path)
    print('Preview video:', single_result.preview_video_path)
    print('Demo overlay:', single_result.demo_overlay_path)
    print('Sheet sync success:', single_result.sheet_sync_success)
    print('Deleted source:', single_result.deleted_source)


In [ ]:
# Show single-run event rows table
if single_result:
    rows = [evt.as_sheet_row() for evt in single_result.events]
    display(pd.DataFrame(rows))

In [ ]:
# Inline video preview (with lines/boxes/IDs/labels)
if SHOW_INLINE_DEMO_VIDEO and single_result and single_result.demo_overlay_path and Path(single_result.demo_overlay_path).exists():
    display(Video(str(single_result.demo_overlay_path), embed=True, html_attributes='controls muted autoplay'))
elif SHOW_INLINE_DEMO_VIDEO and single_result and single_result.preview_video_path and Path(single_result.preview_video_path).exists():
    display(Video(str(single_result.preview_video_path), embed=True, html_attributes='controls muted autoplay'))
else:
    print('No preview video available yet.')

In [ ]:
# Run BATCH from raw folder
batch_results = []
if RUN_BATCH:
    videos = sorted([p for p in RAW_DIR.iterdir() if p.suffix.lower() in {'.mp4', '.mov', '.mkv', '.avi'}])
    print('Batch videos:', len(videos))
    run_prefix = f'master_batch_{datetime.utcnow().strftime("%Y%m%d_%H%M%S")}'
    batch_results = pipeline.process_batch(
        video_paths=videos,
        run_prefix=run_prefix,
        delete_on_success=DELETE_ON_SUCCESS,
        show_live=SHOW_LIVE_POPUP,
    )
    print('Batch complete')
    print(json.dumps(summarize_results(batch_results), indent=2))

In [ ]:
# Batch summary table
if batch_results:
    data = []
    for result in batch_results:
        data.append({
            'video_id': result.metadata.video_id,
            'events': len(result.events),
            'sheet_sync_success': result.sheet_sync_success,
            'deleted_source': result.deleted_source,
            'summary_json': str(result.output_json_path),
            'demo_overlay': str(result.demo_overlay_path) if result.demo_overlay_path else None,
        })
    display(pd.DataFrame(data))

## Notes

- If using Colab GPU kernel in VSCode, this notebook run uses Colab GPU for `.py` pipeline too.
- Live popup windows (`SHOW_LIVE_POPUP=True`) typically work only in local desktop runtime, not headless Colab runtime.
- Demo overlay video still saves and can be viewed inline.
- Source video deletion occurs only when `DELETE_ON_SUCCESS=True` and Google Sheets upsert succeeds.